# Загрузка данных

In [ ]:
# Устанавливаем утилиту для работы с Google Drive
!pip install gdown

# Импортируем библиотеки
import pandas as pd
import gdown

# ссылка
view_url = 'https://drive.google.com/file/d/1YK_SOKFXhLaWdgdQglLxEoAsOMCA7M4x/view?usp=drive_link'

# Извлекаем ID файла из ссылки
file_id = view_url.split('/d/')[1].split('/')[0]
print(f"ID файла: {file_id}")

# Формируем URL для скачивания
download_url = f'https://drive.google.com/uc?id={file_id}'

# Скачиваем файл
output = 'ga_sessions.csv'
gdown.download(download_url, output, quiet=False)

# Загружаем данные в DataFrame
df = pd.read_csv(output)

# Проверяем успешность загрузки
print(f"Файл успешно загружен!")
print(f"Размер данных: {df.shape[0]} строк, {df.shape[1]} столбцов")
print("\nПервые 3 строки:")
df.head(3)

# Базовый анализ и проверка данных

In [ ]:
# Основная информация о данных
print("=== ОСНОВНАЯ ИНФОРМАЦИЯ ===")
print(f"Общий размер: {df.shape[0]:,} строк, {df.shape[1]} столбцов")
print(f"Период данных: с {df['visit_date'].min()} по {df['visit_date'].max()}")

# Проверим типы данных
print("\n=== ТИПЫ ДАННЫХ ===")
print(df.dtypes)

# Проверим пропущенные значения
print("\n=== ПРОПУЩЕННЫЕ ЗНАЧЕНИЯ ===")
missing_data = df.isnull().sum()
missing_percent = (missing_data / len(df)) * 100
missing_info = pd.DataFrame({'Пропусков': missing_data, 'Процент': missing_percent.round(2)})
print(missing_info[missing_info['Пропусков'] > 0])

# Анализ ключевых метрик
print("\n=== КЛЮЧЕВЫЕ МЕТРИКИ ===")
print(f"Уникальных клиентов: {df['client_id'].nunique():,}")
print(f"Уникальных сессий: {df['session_id'].nunique():,}")
print(f"Среднее количество визитов на клиента: {df['visit_number'].mean():.2f}")

In [ ]:
# Удаляем временные колонки, добавленные в предыдущих запусках
columns_to_drop = ['visit_hour', 'visit_datetime', 'month', 'day_of_week',
                   'hour', 'day_of_week_num', 'week', 'is_weekend', 'time_of_day']
df = df.drop(columns=[col for col in columns_to_drop if col in df.columns])

print("Оставшиеся колонки после очистки:")
print(df.columns.tolist())
print(f"\nРазмер данных после очистки: {df.shape}")

# Теперь правильно преобразуем временные метки
print("\n=== ПРЕОБРАЗОВАНИЕ ВРЕМЕННЫХ МЕТОК ===")

# Преобразуем visit_date в datetime
df['visit_date'] = pd.to_datetime(df['visit_date'])
print(f"visit_date преобразован в datetime")

# Создаем полную datetime метку
df['visit_datetime'] = pd.to_datetime(df['visit_date'].astype(str) + ' ' + df['visit_time'])
print(f"visit_datetime создан")

# Извлекаем час из времени
df['visit_hour'] = df['visit_datetime'].dt.hour
print(f"visit_hour извлечен")

# Проверяем результаты
print("\n=== ПРОВЕРКА ПРЕОБРАЗОВАНИЙ ===")
print(f"Диапазон дат: с {df['visit_datetime'].min()} по {df['visit_datetime'].max()}")
print(f"Уникальных дней: {df['visit_date'].nunique()}")
print(f"Пример часов: {df['visit_hour'].unique()[:10]}")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Настройка визуализаций
plt.style.use('default')
sns.set_palette("husl")

print("=== БАЗОВЫЙ АНАЛИЗ ДАННЫХ ===")

# 1. Анализ распределения по устройствам
print("\n1. РАСПРЕДЕЛЕНИЕ ПО УСТРОЙСТВАМ:")
device_counts = df['device_category'].value_counts()
for device, count in device_counts.items():
    percent = count / len(df) * 100
    print(f"  {device}: {count:,} ({percent:.1f}%)")

# 2. Анализ географического распределения
print("\n2. ГЕОГРАФИЧЕСКОЕ РАСПРЕДЕЛЕНИЕ:")
print(f"Стран: {df['geo_country'].nunique()}")
print(f"Городов: {df['geo_city'].nunique()}")

# Топ-5 стран
country_counts = df['geo_country'].value_counts().head(5)
print("Топ-5 стран:")
for country, count in country_counts.items():
    percent = count / len(df) * 100
    print(f"  {country}: {count:,} ({percent:.1f}%)")

# 3. Анализ поведения пользователей
print("\n3. ПОВЕДЕНИЕ ПОЛЬЗОВАТЕЛЕЙ:")
print(f"Уникальных клиентов: {df['client_id'].nunique():,}")
print(f"Всего сессий: {len(df):,}")
print(f"Соотношение сессий/клиентов: {len(df)/df['client_id'].nunique():.2f}")

# Распределение количества визитов
visit_stats = df.groupby('client_id')['visit_number'].max().describe()
print(f"\nРаспределение визитов на клиента:")
print(f"  Среднее: {visit_stats['mean']:.2f}")
print(f"  Медиана: {visit_stats['50%']:.2f}")
print(f"  Максимум: {visit_stats['max']:.0f}")

# Создаем визуализации
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# 1. Распределение по устройствам
device_counts.plot(kind='bar', ax=axes[0, 0], color='skyblue')
axes[0, 0].set_title('Распределение по типам устройств')
axes[0, 0].set_ylabel('Количество визитов')
axes[0, 0].tick_params(axis='x', rotation=45)

# 2. Топ-10 городов
top_cities = df['geo_city'].value_counts().head(10)
top_cities.plot(kind='barh', ax=axes[0, 1], color='lightgreen')
axes[0, 1].set_title('Топ-10 городов по трафику')
axes[0, 1].set_xlabel('Количество визитов')

# 3. Распределение визитов по часам
hourly_dist = df['visit_hour'].value_counts().sort_index()
hourly_dist.plot(kind='line', ax=axes[1, 0], marker='o', color='orange')
axes[1, 0].set_title('Распределение визитов по часам суток')
axes[1, 0].set_xlabel('Час дня')
axes[1, 0].set_ylabel('Количество визитов')
axes[1, 0].grid(True)

# 4. Распределение количества визитов на клиента
visit_distribution = df.groupby('client_id')['visit_number'].max()
visit_distribution[visit_distribution <= 10].hist(bins=10, ax=axes[1, 1], alpha=0.7)
axes[1, 1].set_title('Распределение визитов на клиента (до 10 визитов)')
axes[1, 1].set_xlabel('Количество визитов')
axes[1, 1].set_ylabel('Количество клиентов')

plt.tight_layout()
plt.show()

# Дополнительная информация
print("\n=== ДОПОЛНИТЕЛЬНАЯ ИНФОРМАЦИЯ ===")
print(f"Всего визитов: {len(df):,}")
print(f"Период: {df['visit_date'].min().date()} - {df['visit_date'].max().date()}")
print(f"Уникальных дней: {df['visit_date'].nunique()}")

# Глубокий анализ данных

In [ ]:
print("=== ДЕТАЛЬНЫЙ АНАЛИЗ ПОВЕДЕНИЯ И КАНАЛОВ ===")

# 1. Анализ новых vs возвращающихся пользователей
print("\n1. НОВЫЕ VS ВОЗВРАЩАЮЩИЕСЯ ПОЛЬЗОВАТЕЛИ:")
first_visit = df[df['visit_number'] == 1]
return_visit = df[df['visit_number'] > 1]

print(f"Первых визитов: {len(first_visit):,} ({len(first_visit)/len(df)*100:.1f}%)")
print(f"Повторных визитов: {len(return_visit):,} ({len(return_visit)/len(df)*100:.1f}%)")

# Более детальное распределение
visit_distribution = df.groupby('client_id')['visit_number'].max().value_counts()
print(f"\nДетальное распределение визитов:")
for visits, count in visit_distribution.head(10).items():
    percent = count / df['client_id'].nunique() * 100
    print(f"  {visits} визит(ов): {count:,} клиентов ({percent:.1f}%)")

# 2. Анализ каналов привлечения
print("\n2. АНАЛИЗ КАНАЛОВ ПРИВЛЕЧЕНИЯ:")

# Анализ utm_medium
print("\nРаспределение по UTM Medium:")
medium_stats = df['utm_medium'].value_counts()
for medium, count in medium_stats.items():
    percent = count / len(df) * 100
    print(f"  {medium}: {count:,} ({percent:.1f}%)")

# Анализ топ источников
print("\nТоп-10 источников трафика:")
top_sources = df['utm_source'].value_counts().head(10)
for source, count in top_sources.items():
    percent = count / len(df) * 100
    print(f"  {source}: {count:,} ({percent:.1f}%)")

# 3. Анализ эффективности каналов (ПРАВИЛЬНЫЙ подход)
print("\n3. ЭФФЕКТИВНОСТЬ КАНАЛОВ ПРИВЛЕЧЕНИЯ:")

# Находим максимальное количество визитов для каждого клиента
client_max_visits = df.groupby('client_id')['visit_number'].max().reset_index()

# Находим источник первого визита для каждого клиента
first_visits = df[df['visit_number'] == 1][['client_id', 'utm_source']]
first_visits = first_visits.drop_duplicates('client_id')

# Объединяем данные
client_engagement = client_max_visits.merge(first_visits, on='client_id', how='left')

# Анализируем среднюю глубину визитов по источникам
engagement_by_source = client_engagement.groupby('utm_source')['visit_number'].agg(['mean', 'count']).sort_values('mean', ascending=False)
engagement_by_source = engagement_by_source[engagement_by_source['count'] > 100]  # Фильтруем малочисленные источники

print("Топ-10 каналов по вовлечению (среднее количество визитов на клиента):")
for source, row in engagement_by_source.head(10).iterrows():
    print(f"  {source}: {row['mean']:.2f} визитов ({row['count']:,} клиентов)")

# 4. Визуализация
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# 1. Новые vs возвращающиеся
labels = ['Первые визиты', 'Повторные визиты']
sizes = [len(first_visit), len(return_visit)]
colors = ['#ff9999', '#66b3ff']
axes[0, 0].pie(sizes, labels=labels, colors=colors, autopct='%1.1f%%', startangle=90)
axes[0, 0].set_title('Распределение: новые vs возвращающиеся визиты')

# 2. Топ источники трафика
top_sources.plot(kind='bar', ax=axes[0, 1], color='lightblue')
axes[0, 1].set_title('Топ-10 источников трафика')
axes[0, 1].set_ylabel('Количество визитов')
axes[0, 1].tick_params(axis='x', rotation=45)

# 3. Распределение количества визитов (логарифмическая шкала)
visit_counts = df.groupby('client_id')['visit_number'].max()
visit_counts[visit_counts <= 20].hist(bins=20, ax=axes[1, 0], alpha=0.7)
axes[1, 0].set_title('Распределение визитов на клиента (до 20 визитов)')
axes[1, 0].set_xlabel('Количество визитов')
axes[1, 0].set_ylabel('Количество клиентов')

# 4. Эффективность каналов (топ-10 по вовлечению)
engagement_by_source.head(10)['mean'].plot(kind='bar', ax=axes[1, 1], color='lightgreen')
axes[1, 1].set_title('Топ-10 каналов по вовлечению (средние визиты на клиента)')
axes[1, 1].set_ylabel('Среднее количество визитов')
axes[1, 1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

# Ключевые выводы
print("\n=== КЛЮЧЕВЫЕ ВЫВОДЫ ===")
print(f"• Мобильный трафик доминирует: {79.3}%")
print(f"• Россия составляет: {96.8}% трафика")
print(f"• {len(first_visit)/len(df)*100:.1f}% - первые визиты")
print(f"• {len(return_visit)/len(df)*100:.1f}% - возвращающиеся пользователи")
print(f"• Среднее количество визитов на клиента: {df.groupby('client_id')['visit_number'].max().mean():.2f}")

АНАЛИЗ ВРЕМЕННЫХ ПАТТЕРНОВ И СЕЗОННОСТИ

In [ ]:
print("=== АНАЛИЗ ВРЕМЕННЫХ ПАТТЕРНОВ И СЕЗОННОСТИ ===")

# 1. Создаем дополнительные временные метки
df['visit_month'] = df['visit_datetime'].dt.to_period('M')
df['visit_week'] = df['visit_datetime'].dt.to_period('W')
df['day_of_week'] = df['visit_datetime'].dt.day_name()
df['is_weekend'] = df['day_of_week'].isin(['Saturday', 'Sunday'])

# 2. Анализ по месяцам
print("\n1. АНАЛИЗ ПО МЕСЯЦАМ:")
monthly_traffic = df.groupby('visit_month').size()
print("Трафик по месяцам:")
for month, count in monthly_traffic.items():
    percent = count / len(df) * 100
    print(f"  {month}: {count:,} визитов ({percent:.1f}%)")

# 3. Анализ по неделям
print(f"\n2. АНАЛИЗ ПО НЕДЕЛЯМ:")
print(f"Всего недель: {df['visit_week'].nunique()}")
weekly_stats = df.groupby('visit_week').size().describe()
print(f"Среднее визитов в неделю: {weekly_stats['mean']:.0f}")
print(f"Максимальная неделя: {weekly_stats['max']:.0f} визитов")
print(f"Минимальная неделя: {weekly_stats['min']:.0f} визитов")

# 4. Анализ по дням недели
print(f"\n3. АНАЛИЗ ПО ДНЯМ НЕДЕЛИ:")
dow_traffic = df.groupby('day_of_week').size()
days_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
dow_traffic = dow_traffic.reindex(days_order)

for day, count in dow_traffic.items():
    percent = count / len(df) * 100
    print(f"  {day}: {count:,} визитов ({percent:.1f}%)")

# 5. Анализ выходных vs будни
print(f"\n4. ВЫХОДНЫЕ VS БУДНИ:")
weekday_traffic = len(df[~df['is_weekend']])
weekend_traffic = len(df[df['is_weekend']])
print(f"Будни: {weekday_traffic:,} визитов ({weekday_traffic/len(df)*100:.1f}%)")
print(f"Выходные: {weekend_traffic:,} визитов ({weekend_traffic/len(df)*100:.1f}%)")

# 6. Анализ по времени суток
print(f"\n5. АНАЛИЗ ПО ВРЕМЕНИ СУТОК:")
df['time_of_day'] = pd.cut(df['visit_hour'],
                          bins=[0, 6, 12, 18, 24],
                          labels=['Ночь (0-6)', 'Утро (6-12)', 'День (12-18)', 'Вечер (18-24)'])

time_traffic = df['time_of_day'].value_counts()
for time_period, count in time_traffic.items():
    percent = count / len(df) * 100
    print(f"  {time_period}: {count:,} визитов ({percent:.1f}%)")

# 7. Визуализация временных паттернов
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# 7.1 Трафик по месяцам
monthly_traffic.plot(kind='bar', ax=axes[0, 0], color='skyblue')
axes[0, 0].set_title('Трафик по месяцам')
axes[0, 0].set_ylabel('Количество визитов')
axes[0, 0].tick_params(axis='x', rotation=45)

# 7.2 Трафик по дням недели
dow_traffic.plot(kind='bar', ax=axes[0, 1], color='lightgreen')
axes[0, 1].set_title('Трафик по дням недели')
axes[0, 1].set_ylabel('Количество визитов')
axes[0, 1].tick_params(axis='x', rotation=45)

# 7.3 Трафик по времени суток
time_traffic.plot(kind='bar', ax=axes[1, 0], color='orange')
axes[1, 0].set_title('Трафик по времени суток')
axes[1, 0].set_ylabel('Количество визитов')
axes[1, 0].tick_params(axis='x', rotation=45)

# 7.4 Heatmap: день недели vs час
heatmap_data = df.groupby(['day_of_week', 'visit_hour']).size().unstack(fill_value=0)
heatmap_data = heatmap_data.reindex(days_order)
sns.heatmap(heatmap_data, cmap='YlOrRd', ax=axes[1, 1], cbar_kws={'label': 'Количество визитов'})
axes[1, 1].set_title('Трафик: день недели vs час')
axes[1, 1].set_xlabel('Час дня')
axes[1, 1].set_ylabel('День недели')

plt.tight_layout()
plt.show()

# 8. Анализ роста трафика
print(f"\n6. АНАЛИЗ РОСТА ТРАФИКА:")
daily_traffic = df.groupby('visit_date').size()
cumulative_traffic = daily_traffic.cumsum()

print(f"Среднедневной трафик: {daily_traffic.mean():.0f} визитов")
print(f"Максимальный день: {daily_traffic.max():.0f} визитов")
print(f"Минимальный день: {daily_traffic.min():.0f} визитов")

# Визуализация кумулятивного роста
plt.figure(figsize=(12, 6))
plt.plot(cumulative_traffic.index, cumulative_traffic.values, linewidth=2, color='blue')
plt.title('Кумулятивный рост трафика')
plt.xlabel('Дата')
plt.ylabel('Общее количество визитов')
plt.grid(True)
plt.show()

# Ключевые выводы по временным паттернам
print("\n=== КЛЮЧЕВЫЕ ВЫВОДЫ ПО ВРЕМЕННЫМ ПАТТЕРНАМ ===")
print(f"• Всего месяцев: {df['visit_month'].nunique()}")
print(f"• Всего недель: {df['visit_week'].nunique()}")
print(f"• Пиковое время суток: {time_traffic.idxmax()}")
print(f"• Самый активный день недели: {dow_traffic.idxmax()}")
print(f"• Соотношение будни/выходные: {weekday_traffic/weekend_traffic:.2f}:1")

АНАЛИЗ ТЕХНИЧЕСКИХ ХАРАКТЕРИСТИК И ГЕОГРАФИИ

In [ ]:
print("=== АНАЛИЗ ТЕХНИЧЕСКИХ ХАРАКТЕРИСТИК И ГЕОГРАФИИ ===")

# 1. Анализ технических характеристик (только с заполненными данными)
print("\n1. ТЕХНИЧЕСКИЕ ХАРАКТЕРИСТИКИ:")

# Анализ устройств с заполненными данными
devices_with_data = df[df['device_brand'].notna()]
print(f"Записей с данными об устройствах: {len(devices_with_data):,} ({len(devices_with_data)/len(df)*100:.1f}%)")

if len(devices_with_data) > 0:
    print(f"\nТоп-10 брендов устройств:")
    brand_counts = devices_with_data['device_brand'].value_counts().head(10)
    for brand, count in brand_counts.items():
        percent = count / len(devices_with_data) * 100
        print(f"  {brand}: {count:,} ({percent:.1f}%)")

    print(f"\nТоп-10 операционных систем:")
    os_counts = devices_with_data['device_os'].value_counts().head(10)
    for os, count in os_counts.items():
        percent = count / len(devices_with_data) * 100
        print(f"  {os}: {count:,} ({percent:.1f}%)")

# Анализ браузеров
print(f"\nТоп-10 браузеров:")
browser_counts = df['device_browser'].value_counts().head(10)
for browser, count in browser_counts.items():
    percent = count / len(df) * 100
    print(f"  {browser}: {count:,} ({percent:.1f}%)")

# Анализ разрешений экранов
print(f"\nТоп-10 разрешений экранов:")
resolution_counts = df['device_screen_resolution'].value_counts().head(10)
for res, count in resolution_counts.items():
    percent = count / len(df) * 100
    print(f"  {res}: {count:,} ({percent:.1f}%)")

# 2. Глубокий географический анализ
print("\n2. ГЛУБОКИЙ ГЕОГРАФИЧЕСКИЙ АНАЛИЗ:")

# Анализ по России (96.8% трафика)
russian_traffic = df[df['geo_country'] == 'Russia']
print(f"Российский трафик: {len(russian_traffic):,} визитов")

print(f"\nТоп-20 городов России:")
russian_cities = russian_traffic['geo_city'].value_counts().head(20)
for city, count in russian_cities.items():
    percent = count / len(russian_traffic) * 100
    print(f"  {city}: {count:,} ({percent:.1f}%)")

# Анализ международного трафика
international_traffic = df[df['geo_country'] != 'Russia']
print(f"\nМеждународный трафик: {len(international_traffic):,} визитов ({len(international_traffic)/len(df)*100:.1f}%)")

if len(international_traffic) > 0:
    print(f"Топ-10 стран (кроме России):")
    intl_countries = international_traffic['geo_country'].value_counts().head(10)
    for country, count in intl_countries.items():
        percent = count / len(international_traffic) * 100
        print(f"  {country}: {count:,} ({percent:.1f}%)")

# 3. Сравнение технических характеристик по географии
print("\n3. СРАВНЕНИЕ ТЕХНИЧЕСКИХ ХАРАКТЕРИСТИК:")

# Сравнение устройств: Россия vs международный трафик
if len(international_traffic) > 0:
    print("Распределение устройств (Россия vs международный):")
    ru_devices = russian_traffic['device_category'].value_counts(normalize=True) * 100
    intl_devices = international_traffic['device_category'].value_counts(normalize=True) * 100

    for device in ru_devices.index:
        ru_percent = ru_devices[device]
        intl_percent = intl_devices.get(device, 0)
        print(f"  {device}: Россия {ru_percent:.1f}% vs Международный {intl_percent:.1f}%")

# 4. Визуализация
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# 4.1 Топ бренды устройств
if len(devices_with_data) > 0:
    brand_counts.head(10).plot(kind='bar', ax=axes[0, 0], color='lightblue')
    axes[0, 0].set_title('Топ-10 брендов устройств')
    axes[0, 0].set_ylabel('Количество устройств')
    axes[0, 0].tick_params(axis='x', rotation=45)

# 4.2 Топ операционные системы
if len(devices_with_data) > 0:
    os_counts.head(10).plot(kind='bar', ax=axes[0, 1], color='lightgreen')
    axes[0, 1].set_title('Топ-10 операционных систем')
    axes[0, 1].set_ylabel('Количество устройств')
    axes[0, 1].tick_params(axis='x', rotation=45)

# 4.3 Топ города России
russian_cities.head(10).plot(kind='barh', ax=axes[1, 0], color='orange')
axes[1, 0].set_title('Топ-10 городов России')
axes[1, 0].set_xlabel('Количество визитов')

# 4.4 Сравнение устройств Россия vs международный
if len(international_traffic) > 0:
    comparison_data = pd.DataFrame({
        'Russia': ru_devices,
        'International': intl_devices
    }).fillna(0)
    comparison_data.plot(kind='bar', ax=axes[1, 1])
    axes[1, 1].set_title('Распределение устройств: Россия vs Международный')
    axes[1, 1].set_ylabel('Процент')
    axes[1, 1].tick_params(axis='x', rotation=45)
    axes[1, 1].legend()

plt.tight_layout()
plt.show()

# 5. Анализ качества данных
print("\n4. АНАЛИЗ КАЧЕСТВА ДАННЫХ:")
print("Заполненность технических данных:")
tech_columns = ['device_brand', 'device_os', 'device_model', 'device_screen_resolution']
for col in tech_columns:
    filled = df[col].notna().sum()
    percent = filled / len(df) * 100
    print(f"  {col}: {filled:,} ({percent:.1f}%)")

# Ключевые выводы
print("\n=== КЛЮЧЕВЫЕ ВЫВОДЫ ===")
print(f"• Заполненность данных об устройствах: {len(devices_with_data)/len(df)*100:.1f}%")
print(f"• Топ город России: {russian_cities.index[0] if len(russian_cities) > 0 else 'N/A'}")
print(f"• Международный трафик: {len(international_traffic)/len(df)*100:.1f}%")
print(f"• Топ браузер: {browser_counts.index[0] if len(browser_counts) > 0 else 'N/A'}")

# Финальный анализ и обобщение результатов

In [ ]:
print("=== ФИНАЛЬНЫЙ АНАЛИЗ И ОБОБЩЕНИЕ РЕЗУЛЬТАТОВ ===")

# 1. Создаем сводную таблицу ключевых метрик
print("\n1. КЛЮЧЕВЫЕ МЕТРИКИ ПРОЕКТА:")
metrics = {
    'Общее количество визитов': f"{len(df):,}",
    'Уникальных клиентов': f"{df['client_id'].nunique():,}",
    'Уникальных сессий': f"{df['session_id'].nunique():,}",
    'Период данных': f"{df['visit_date'].min().date()} - {df['visit_date'].max().date()}",
    'Среднее визитов на клиента': f"{df.groupby('client_id')['visit_number'].max().mean():.2f}",
    'Процент возвращающихся': f"{(len(df[df['visit_number'] > 1])/len(df)*100):.1f}%",
    'Мобильный трафик': f"{(len(df[df['device_category'] == 'mobile'])/len(df)*100):.1f}%",
    'Российский трафик': f"{(len(df[df['geo_country'] == 'Russia'])/len(df)*100):.1f}%"
}

for metric, value in metrics.items():
    print(f"  {metric}: {value}")

# 2. Анализ самых ценных каналов
print("\n2. АНАЛИЗ ЭФФЕКТИВНОСТИ КАНАЛОВ:")

# Находим топ каналы по количеству привлеченных клиентов
top_channels = df[df['visit_number'] == 1]['utm_source'].value_counts().head(10)
print("Топ-10 каналов по привлечению новых клиентов:")
for channel, count in top_channels.items():
    percent = count / len(df[df['visit_number'] == 1]) * 100
    print(f"  {channel}: {count:,} клиентов ({percent:.1f}%)")

# 3. Анализ retention по каналам
print("\n3. АНАЛИЗ RETENTION ПО КАНАЛАМ:")

# Для каждого канала считаем процент возвращающихся пользователей
retention_by_channel = {}
for channel in top_channels.index:
    channel_clients = df[df['utm_source'] == channel]['client_id'].unique()
    if len(channel_clients) > 100:  # только каналы с достаточным объемом
        returning = df[(df['client_id'].isin(channel_clients)) & (df['visit_number'] > 1)]
        retention_rate = len(returning['client_id'].unique()) / len(channel_clients) * 100
        retention_by_channel[channel] = retention_rate

# Сортируем по retention rate
retention_by_channel = dict(sorted(retention_by_channel.items(), key=lambda x: x[1], reverse=True))

print("Retention rate по каналам (топ-5):")
for channel, retention in list(retention_by_channel.items())[:5]:
    print(f"  {channel}: {retention:.1f}%")

# 4. Сезонность и тренды
print("\n4. СЕЗОННОСТЬ И ТРЕНДЫ:")

# Анализ роста по месяцам
monthly_growth = df.groupby('visit_month').size().pct_change() * 100
print("Месячный рост трафика:")
for month, growth in monthly_growth.items():
    if not pd.isna(growth):
        print(f"  {month}: {growth:+.1f}%")

# 5. Технологические тренды
print("\n5. ТЕХНОЛОГИЧЕСКИЕ ТРЕНДЫ:")

# Анализ популярности устройств по времени
device_trends = df.groupby(['visit_month', 'device_category']).size().unstack(fill_value=0)
device_trends_pct = device_trends.div(device_trends.sum(axis=1), axis=0) * 100

print("Динамика доли мобильного трафика:")
for month in device_trends_pct.index:
    mobile_pct = device_trends_pct.loc[month, 'mobile']
    print(f"  {month}: {mobile_pct:.1f}%")

# 6. Финальные визуализации
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# 6.1 Ключевые метрики
metrics_names = list(metrics.keys())[:4]
metrics_values = [int(metrics[m].replace(',', '')) for m in metrics_names[:3]]
metrics_values.append(226)  # количество дней

axes[0, 0].bar(metrics_names, metrics_values, color=['blue', 'green', 'orange', 'red'])
axes[0, 0].set_title('Ключевые метрики проекта')
axes[0, 0].tick_params(axis='x', rotation=45)
for i, v in enumerate(metrics_values):
    axes[0, 0].text(i, v, str(v), ha='center', va='bottom')

# 6.2 Распределение трафика по источникам
top_sources = df['utm_source'].value_counts().head(8)
other_sources = len(df) - top_sources.sum()
top_sources['Other'] = other_sources
top_sources.plot(kind='pie', ax=axes[0, 1], autopct='%1.1f%%')
axes[0, 1].set_title('Распределение трафика по источникам')

# 6.3 Динамика мобильного трафика
device_trends_pct.plot(kind='line', ax=axes[1, 0], marker='o')
axes[1, 0].set_title('Динамика доли устройств по месяцам')
axes[1, 0].set_ylabel('Доля, %')
axes[1, 0].grid(True)
axes[1, 0].legend(bbox_to_anchor=(1.05, 1), loc='upper left')

# 6.4 Географическое распределение (Россия)
russian_cities = df[df['geo_country'] == 'Russia']['geo_city'].value_counts().head(10)
russian_cities.plot(kind='barh', ax=axes[1, 1], color='lightblue')
axes[1, 1].set_title('Топ-10 городов России по трафику')
axes[1, 1].set_xlabel('Количество визитов')

plt.tight_layout()
plt.show()

# 7. Рекомендации для бизнеса
print("\n" + "="*50)
print("РЕКОМЕНДАЦИИ ДЛЯ БИЗНЕСА:")
print("="*50)

print("1. 📱 МОБИЛЬНАЯ СТРАТЕГИЯ:")
print("   • 79.3% трафика - мобильные устройства")
print("   • Фокус на mobile-first дизайн и UX")
print("   • Оптимизация для iOS и Android")

print("\n2. 🇷🇺 ГЕОГРАФИЧЕСКИЙ ФОКУС:")
print("   • 96.8% трафика из России")
print("   • Москва и СПб - 61.2% российского трафика")
print("   • Развитие регионального присутствия")

print("\n3. 📊 ЭФФЕКТИВНОСТЬ КАНАЛОВ:")
print("   • Топ-каналы: ZpYIoDJMcFzVoPFsHGJL (31.1%), fDLlAcSmythWSCVMvqvL (16.2%)")
print("   • Увеличить инвестиции в эффективные каналы")
print("   • Анализировать retention по каналам")

print("\n4. 🔄 RETENTION И ЛОЯЛЬНОСТЬ:")
print("   • 82.5% клиентов - одноразовые визиты")
print("   • Разработать стратегию удержания")
print("   • Внедрить программы лояльности")

print("\n5. 📈 СЕЗОННОСТЬ И ТРЕНДЫ:")
print("   • Рост трафика к концу года (21.5% в декабре)")
print("   • Пик активности: вторник, день (12-18)")
print("   • Учитывать сезонность в планировании")

print("\n6. 🛠 ТЕХНИЧЕСКИЕ УЛУЧШЕНИЯ:")
print("   • Оптимизация для Chrome (54.5%) и Safari (25.7%)")
print("   • Поддержка популярных разрешений экранов")
print("   • Улучшение tracking качества данных")

print("\n7. 🌍 МЕЖДУНАРОДНОЕ РАЗВИТИЕ:")
print("   • 3.2% международного трафика")
print("   • Потенциал роста: США, Украина, Европа")
print("   • Локализация для международной аудитории")

СТРАТЕГИЧЕСКИЕ ВЫВОДЫ И ПЛАН ДЕЙСТВИЙ

In [ ]:
print("="*60)
print("СТРАТЕГИЧЕСКИЕ ВЫВОДЫ И ПЛАН ДЕЙСТВИЙ")
print("="*60)

# 1. Ключевые инсайты
print("\n🔍 КЛЮЧЕВЫЕ ИНСАЙТЫ:")
insights = [
    "• Канал 'kjsLglQLzykiRbcDiGcD' показывает наивысший retention (46.8%) при 8.6% охвата",
    "• Резкий рост мобильного трафика с 66.2% в мае до 86.8% в декабре",
    "• 82.5% клиентов совершают только 1 визит - огромный потенциал для роста retention",
    "• Москва и СПб генерируют 61.2% всего российского трафика",
    "• Месячный рост трафика в декабре +34% - сезонный пик"
]

for insight in insights:
    print(insight)

# 2. Приоритетные задачи
print("\n🚀 ПРИОРИТЕТНЫЕ ЗАДАЧИ НА 3 МЕСЯЦА:")

priorities = {
    1: "Увеличить инвестиции в канал 'kjsLglQLzykiRbcDiGcD' (макс. retention)",
    2: "Разработать программу лояльности для 82.5% 'одноразовых' клиентов",
    3: "Запустить mobile-first редизайн (ориентация на 86.8% мобильного трафика)",
    4: "Разработать региональную стратегию (выход за пределы Москвы/СПб)",
    5: "Улучшить tracking качества данных (80.3% заполненность device_brand)"
}

for priority, task in priorities.items():
    print(f"{priority}. {task}")

# 3. Ключевые показатели для мониторинга
print("\n📊 KPI ДЛЯ МОНИТОРИНГА:")
kpis = {
    "Retention Rate": "> 30% (сейчас 26.0%)",
    "Mobile Traffic": "> 85% (сейчас 79.3%)",
    "Regional Coverage": "> 40% вне Москвы/СПб (сейчас 38.8%)",
    "Client LTV": "Увеличение на 25%",
    "Data Quality": "> 90% заполненность технических данных"
}

for kpi, target in kpis.items():
    print(f"• {kpi}: {target}")

# 4. Роадмап реализации
print("\n🗓️ РОАДМАП РЕАЛИЗАЦИИ (3 месяца):")

roadmap = {
    "МЕСЯЦ 1": [
        "Аудит текущих маркетинговых каналов",
        "Разработка MVP программы лояльности",
        "Начало mobile-first редизайна"
    ],
    "МЕСЯЦ 2": [
        "Запуск кампании в топовых каналах",
        "Тестирование программы лояльности",
        "Внедрение улучшенного tracking"
    ],
    "МЕСЯЦ 3": [
        "Масштабирование успешных каналов",
        "Полный запуск программы лояльности",
        "Запуск региональной экспансии"
    ]
}

for month, tasks in roadmap.items():
    print(f"\n{month}:")
    for task in tasks:
        print(f"  • {task}")

# 5. Ожидаемые результаты
print("\n📈 ОЖИДАЕМЫЕ РЕЗУЛЬТАТЫ (через 6 месяцев):")
expected_results = [
    "Увеличение retention rate на 15-20%",
    "Рост LTV клиента на 25-30%",
    "Увеличение доли повторных покупок на 20%",
    "Расширение географии на 15-20 новых городов",
    "Улучшение ROI маркетинговых каналов на 35%"
]

for result in expected_results:
    print(f"• {result}")

# Финальная визуализация стратегии
plt.figure(figsize=(14, 10))

# 1. Приоритеты инвестиций
plt.subplot(2, 2, 1)
channels = ['kjsLglQL...', 'ZpYIoDJMc...', 'fDLlAcSm...', 'BHcvLfOa...', 'bByPQxmD...']
retention_rates = [46.8, 28.4, 28.4, 27.5, 39.6]  # Примерные значения
plt.barh(channels, retention_rates, color=['green', 'blue', 'blue', 'orange', 'red'])
plt.title('Приоритетность каналов по retention rate')
plt.xlabel('Retention Rate (%)')
plt.grid(axis='x')

# 2. Эволюция мобильного трафика
plt.subplot(2, 2, 2)
months = ['May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
mobile_pct = [66.2, 81.9, 66.0, 74.9, 76.6, 83.8, 81.3, 86.8]
plt.plot(months, mobile_pct, marker='o', linewidth=2, color='purple')
plt.title('Рост мобильного трафика (2021)')
plt.ylabel('Доля мобильного трафика (%)')
plt.grid(True)
plt.ylim(60, 90)

# 3. Распределение клиентов по количеству визитов
plt.subplot(2, 2, 3)
visit_distribution = [82.5, 11.3, 3.0, 1.3, 0.6, 2.3]  # 1,2,3,4,5,6+ визитов
labels = ['1 визит', '2 визита', '3 визита', '4 визита', '5 визитов', '6+ визитов']
colors = ['red', 'orange', 'yellow', 'lightgreen', 'green', 'darkgreen']
plt.pie(visit_distribution, labels=labels, colors=colors, autopct='%1.1f%%')
plt.title('Распределение клиентов по визитам')

# 4. Географическое распределение
plt.subplot(2, 2, 4)
geo_distribution = [44.7, 16.5, 38.8]  # Moscow, SPb, Other
labels = ['Москва', 'Санкт-Петербург', 'Другие города']
plt.pie(geo_distribution, labels=labels, autopct='%1.1f%%')
plt.title('Географическое распределение трафика')

plt.tight_layout()
plt.show()

print("\n🎯 ЗАКЛЮЧЕНИЕ:")
print("Данный анализ предоставляет комплексную основу для:")
print("• Оптимизации маркетинговых инвестиций")
print("• Улучшения пользовательского опыта")
print("• Разработки стратегии роста и удержания")
print("• Повышения общей эффективности бизнеса")

# Машинное обучение: кластеризация пользователей

In [ ]:
# Создаем осмысленные признаки для каждого пользователя
user_features = df.groupby('client_id').agg({
    'visit_number': 'max',           # Максимальное количество визитов
    'visit_date': 'nunique',         # Количество уникальных дней с визитами
    'visit_hour': ['mean', 'std'],    # Среднее время визита и его стабильность
    'device_category': lambda x: x.mode()[0] if len(x.mode()) > 0 else 'unknown',
    'utm_source': lambda x: x.mode()[0] if len(x.mode()) > 0 else 'unknown',
    'geo_city': lambda x: x.mode()[0] if len(x.mode()) > 0 else 'unknown'
})

# Переименовываем колонки
user_features.columns = ['max_visits', 'unique_days', 'avg_hour', 'hour_std',
                         'fav_device', 'fav_source', 'fav_city']

# Фильтруем выбросы и NaN
user_features = user_features[(user_features['max_visits'] <= 20) &
                             (user_features['unique_days'] <= 15)]
user_features = user_features.dropna()

print(f"Пользователей для кластеризации: {len(user_features):,}")

# 2. Подготовка числовых признаков
print("2. ПОДГОТОВКА ЧИСЛОВЫХ ПРИЗНАКОВ...")

numeric_features = user_features[['max_visits', 'unique_days', 'avg_hour', 'hour_std']].copy()

# Масштабирование
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
scaled_features = scaler.fit_transform(numeric_features)

# 3. Определение оптимального числа кластеров
print("3. ОПРЕДЕЛЕНИЕ ОПТИМАЛЬНОГО ЧИСЛА КЛАСТЕРОМ...")

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Метод локтя
inertia = []
k_range = range(2, 8)

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(scaled_features)
    inertia.append(kmeans.inertia_)

# Визуализация метода локтя
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(k_range, inertia, marker='o')
plt.xlabel('Number of clusters')
plt.ylabel('Inertia')
plt.title('Elbow Method')

# Метод силуэта
silhouette_scores = []
for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    cluster_labels = kmeans.fit_predict(scaled_features)
    silhouette_avg = silhouette_score(scaled_features, cluster_labels)
    silhouette_scores.append(silhouette_avg)

plt.subplot(1, 2, 2)
plt.plot(k_range, silhouette_scores, marker='o', color='green')
plt.xlabel('Number of clusters')
plt.ylabel('Silhouette Score')
plt.title('Silhouette Method')
plt.tight_layout()
plt.show()

# Выбираем оптимальное количество кластеров
optimal_k = 4
kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
user_features['cluster'] = kmeans.fit_predict(scaled_features)

# 4. Анализ кластеров
print("4. АНАЛИЗ КЛАСТЕРОМ...")

# Статистика по кластерам
cluster_stats = user_features.groupby('cluster').agg({
    'max_visits': ['mean', 'std'],
    'unique_days': ['mean', 'std'],
    'avg_hour': ['mean', 'std'],
    'hour_std': ['mean', 'std']
}).round(2)

print("Статистика по кластерам:")
print(cluster_stats)

# Анализ категориальных признаков по кластерам
print("\nРаспределение устройств по кластерам:")
device_by_cluster = pd.crosstab(user_features['cluster'], user_features['fav_device'], normalize='index') * 100
print(device_by_cluster.round(1))

print("\nРаспределение источников по кластерам:")
source_by_cluster = pd.crosstab(user_features['cluster'], user_features['fav_source'], normalize='index') * 100
print(source_by_cluster[source_by_cluster.max(axis=0) > 5].round(1))  # Только значимые источники

# 5. Визуализация кластеров
print("5. ВИЗУАЛИЗАЦИЯ РЕЗУЛЬТАТОВ...")

fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# 5.1 Распределение по кластерам
cluster_counts = user_features['cluster'].value_counts()
axes[0, 0].pie(cluster_counts.values, labels=[f'Cluster {i}' for i in cluster_counts.index],
               autopct='%1.1f%%', startangle=90)
axes[0, 0].set_title('Распределение пользователей по кластерам')

# 5.2 Среднее количество визитов по кластерам
visits_by_cluster = user_features.groupby('cluster')['max_visits'].mean()
axes[0, 1].bar(visits_by_cluster.index, visits_by_cluster.values, color='lightblue')
axes[0, 1].set_title('Среднее количество визитов по кластерам')
axes[0, 1].set_xlabel('Кластер')
axes[0, 1].set_ylabel('Среднее количество визитов')
for i, v in enumerate(visits_by_cluster.values):
    axes[0, 1].text(i, v, f'{v:.1f}', ha='center', va='bottom')

# 5.3 Среднее время визита по кластерам
hour_by_cluster = user_features.groupby('cluster')['avg_hour'].mean()
axes[1, 0].bar(hour_by_cluster.index, hour_by_cluster.values, color='lightgreen')
axes[1, 0].set_title('Среднее время визита по кластерам')
axes[1, 0].set_xlabel('Кластер')
axes[1, 0].set_ylabel('Средний час визита')
axes[1, 0].set_ylim(0, 24)
for i, v in enumerate(hour_by_cluster.values):
    axes[1, 0].text(i, v, f'{v:.1f}', ha='center', va='bottom')

# 5.4 Распределение устройств по кластерам (heatmap)
sns.heatmap(device_by_cluster, annot=True, fmt='.1f', cmap='YlOrRd', ax=axes[1, 1])
axes[1, 1].set_title('Распределение устройств по кластерам (%)')
axes[1, 1].set_xlabel('Тип устройства')
axes[1, 1].set_ylabel('Кластер')

plt.tight_layout()
plt.show()

# 6. Интерпретация кластеров
print("6. ИНТЕРПРЕТАЦИЯ КЛАСТЕРОМ:")

cluster_descriptions = {
    0: "ОДНОРАЗОВЫЕ ПОСЕТИТЕЛИ - 1-2 визита, разное время, в основном мобильные",
    1: "АКТИВНЫЕ ДНЕВНЫЕ ПОЛЬЗОВАТЕЛИ - 3-5 визитов, дневное время, mobile/desktop",
    2: "ВЕЧЕРНИЕ ПОЛЬЗОВАТЕЛИ - 2-3 визита, вечернее время, преимущественно mobile",
    3: "ПОСТОЯННЫЕ КЛИЕНТЫ - 5+ визитов, стабильное время, разнообразные устройства"
}

print("\nОПИСАНИЕ КЛАСТЕРОМ:")
for cluster_id, description in cluster_descriptions.items():
    cluster_size = len(user_features[user_features['cluster'] == cluster_id])
    cluster_percent = cluster_size / len(user_features) * 100
    print(f"Кластер {cluster_id} ({cluster_percent:.1f}%): {description}")

# 7. Рекомендации для каждого кластера
print("\n7. РЕКОМЕНДАЦИИ ПО КЛАСТЕРАМ:")

cluster_recommendations = {
    0: "• Программа welcome-бонусов\n• Упрощенная регистрация\n• Персональные предложения",
    1: "• Программа лояльности\n• Регулярные уведомления\n• Эксклюзивный контент",
    2: "• Вечерние акции\n• Push-уведомления в 18-20 часов\n• Мобильные предложения",
    3: "• VIP-программа\n• Персональный менеджер\n• Специальные условия"
}

for cluster_id, recommendations in cluster_recommendations.items():
    print(f"\nКластер {cluster_id}:")
    for recommendation in recommendations.split('\n'):
        print(f"  {recommendation}")